In [ ]:
!pip install -q transformers datasets peft bitsandbytes accelerate evaluate wandb

In [ ]:
from google.colab import userdata
from huggingface_hub import login
import wandb, os

login(token=userdata.get('HF_TOKEN'))
wandb.login(key=userdata.get('WANDB_API_KEY'))

In [ ]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

In [ ]:
from transformers import AutoModelForSequenceClassification, BitsAndBytesConfig
import torch

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "meta-llama/Llama-3.2-1B",
    num_labels=5,
    quantization_config=bnb_config,
    id2label={0: "billing", 1: "technical", 2: "account", 3: "shipping", 4: "general"},
    label2id={"billing": 0, "technical": 1, "account": 2, "shipping": 3, "general": 4},
)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
model.config.pad_token_id = tokenizer.pad_token_id

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS",
)

In [ ]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
from datasets import load_from_disk
import numpy as np
import evaluate

In [ ]:
dataset = load_from_disk("/content/drive/MyDrive/processed")
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1.compute(predictions=preds, references=labels, average="macro")["f1"],
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/ticket-classifier/outputs/ticket-classifier-lora",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    learning_rate=2e-4,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    report_to="wandb",
    run_name="ticket-classifier-lora-prompt",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    seed=42
)

In [ ]:
import os
os.makedirs("/content/drive/MyDrive/ticket-classifier/outputs", exist_ok=True)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
best_checkpoint = trainer.state.best_model_checkpoint

In [ ]:
metrics = trainer.evaluate()
print(metrics)

In [ ]:
model.push_to_hub("sheethal00/ticket-classifier-lora")
tokenizer.push_to_hub("sheethal00/ticket-classifier-lora")

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

predictions = trainer.predict(dataset["validation"])
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

label_names = ["billing", "technical", "account", "shipping", "general"]

print(classification_report(labels, preds, target_names=label_names))

# Save to Drive so this survives a session drop
save_dir = "/content/drive/MyDrive/ticket-classifier/eval_outputs/"
import os
os.makedirs(save_dir, exist_ok=True)

np.save(save_dir + "preds.npy", preds)
np.save(save_dir + "labels.npy", labels)

# Also save the raw logits, in case you want confidence scores later (e.g. for the "general as fallback" idea)
np.save(save_dir + "logits.npy", predictions.predictions)

cm = confusion_matrix(labels, preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=label_names, yticklabels=label_names, cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix — Validation Set")
plt.show()

In [ ]:
import pandas as pd

label_names = ["billing", "technical", "account", "shipping", "general"]

# NOTE: if you removed the "text" column during tokenization for dataset["validation"],
# reload the untokenized version just for this — see note below
val_raw = dataset["validation"]  # adjust if text column was removed, see note

wrong_idx = np.where(preds != labels)[0]
df = pd.DataFrame({
    "text": [val_raw[int(i)]["text"] for i in wrong_idx],
    "true": [label_names[labels[i]] for i in wrong_idx],
    "pred": [label_names[preds[i]] for i in wrong_idx],
})

df.to_csv(save_dir + "misclassified.csv", index=False)
df.to_json(save_dir + "misclassified.json", orient="records", indent=2)

print(f"Saved {len(df)} misclassified examples to Drive")

In [ ]:
from transformers import AutoModelForSequenceClassification, BitsAndBytesConfig
import torch
from peft import PeftModel

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

id2label= {0: "billing", 1: "technical", 2: "account", 3: "shipping", 4: "general"}
label2id= {"billing": 0, "technical": 1, "account": 2, "shipping": 3, "general": 4}

base_model = AutoModelForSequenceClassification.from_pretrained(
    "meta-llama/Llama-3.2-1B",
    num_labels=5,
    quantization_config=bnb_config,
    id2label=id2label, label2id=label2id,
)

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model.config.pad_token_id = tokenizer.pad_token_id

model = PeftModel.from_pretrained(base_model, best_checkpoint)
model.eval()

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

eval_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/ticket-classifier/eval_only",  # not really used for saving, just required
    per_device_eval_batch_size=8,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=eval_args,
    eval_dataset=dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
predictions = trainer.predict(dataset["validation"])
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

print(predictions.metrics)